# STG-NF YOLO26-pose Extraction, Training, and Testing on Colab

Identical flow to `STG-NF.ipynb`, but poses are extracted with **Ultralytics YOLO26-pose**
(a single detection + pose model) instead of AlphaPose, and written to a **new**
`pose_yolopose` feature folder so existing AlphaPose poses are left untouched.

Training, data-loading, ground-truth prep, and score export are byte-for-byte the same.


## 1. Mount Drive and Configure Paths

Set **`DATASET_NAME`** to `"shanghaitech"` or `"avenue"`. The cell below resolves archive paths, local extract roots, source folders, ground-truth labels, and the Drive output root.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import time

import torch

assert torch.cuda.is_available(), "No GPU found. Use Runtime -> Change runtime type -> GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)

drive.mount("/content/drive")

REPO_URL = "https://github.com/Hadi6618/STG-NF.git"
REPO_DIR = Path("/content/STG-NF")
LOCAL_POSE_WORK = Path("/content/stg_nf_yolopose_work")

# --- Dataset selection: "shanghaitech" or "avenue" ---
# Avenue expects MyDrive/Avenue_Dataset.zip and MyDrive/ground_truth_avenue/{1..21}.npy
DATASET_NAME = "avenue"

DATASET_PRESETS = {
    "shanghaitech": {
        "display_name": "ShanghaiTech Campus",
        "stg_nf_dataset_arg": "ShanghaiTech",
        "archive_path": Path("/content/drive/MyDrive/shanghaitech.tar.gz"),
        "extract_root": Path("/content"),
        "extract_marker": Path("/content/shanghaitech/training/videos"),
        "archive_is_zip": False,
        "original_data_root": Path("/content/shanghaitech"),
        "train_relative": Path("training/videos"),
        "test_relative": Path("testing/frames"),
        "train_source_mode": "video",
        "test_source_mode": "images",
        "drive_root": Path("/content/drive/MyDrive/STG-NF/original_shanghaitech"),
        "ground_truth_dir": None,
        "gt_repo_subdir": Path("data/ShanghaiTech/gt/test_frame_mask"),
        "expected_train": 330,
        "expected_test": 107,
    },
    "avenue": {
        "display_name": "CUHK Avenue",
        "stg_nf_dataset_arg": "Avenue",
        "archive_path": Path("/content/drive/MyDrive/Avenue_Dataset.zip"),
        "extract_root": Path("/content/avenue_dataset"),
        "extract_marker": Path("/content/avenue_dataset/Avenue Dataset/training_videos"),
        "archive_is_zip": True,
        "original_data_root": Path("/content/avenue_dataset/Avenue Dataset"),
        "train_relative": Path("training_videos"),
        "test_relative": Path("testing_videos"),
        "train_source_mode": "video",
        "test_source_mode": "video",
        "drive_root": Path("/content/drive/MyDrive/STG-NF/Avenue_dataset_Confg_0_5"),
        "ground_truth_dir": Path("/content/drive/MyDrive/ground_truth_avenue"),
        "gt_repo_subdir": Path("data/Avenue/gt/test_frame_mask"),
        "expected_train": 16,
        "expected_test": 21,
    },
}

DATASET_KEY = DATASET_NAME.lower().strip()
if DATASET_KEY not in DATASET_PRESETS:
    raise ValueError(
        f"Unsupported DATASET_NAME={DATASET_NAME!r}. Choose one of: {sorted(DATASET_PRESETS)}"
    )

DATASET_CONFIG = DATASET_PRESETS[DATASET_KEY]
DISPLAY_DATASET_NAME = DATASET_CONFIG["display_name"]
STG_NF_DATASET_ARG = DATASET_CONFIG["stg_nf_dataset_arg"]

ORIGINAL_DATA_ROOT = Path(DATASET_CONFIG["original_data_root"])
TRAIN_SOURCE_ROOT = ORIGINAL_DATA_ROOT / DATASET_CONFIG["train_relative"]
TEST_SOURCE_ROOT = ORIGINAL_DATA_ROOT / DATASET_CONFIG["test_relative"]
TRAIN_SOURCE_MODE = DATASET_CONFIG["train_source_mode"]
TEST_SOURCE_MODE = DATASET_CONFIG["test_source_mode"]

# YOLO26-pose extraction knobs (single detection+pose model replaces the AlphaPose pipeline).
POSE_MODEL_NAME = "yolo26l-pose.pt"   # Ultralytics YOLO26-pose (detects person + 17 COCO keypoints)
DET_THR  = 0.3     # person detection confidence threshold
TRACKER  = "bytetrack.yaml"   # Ultralytics built-in tracker: "bytetrack.yaml" or "botsort.yaml"

DRIVE_ROOT = Path(DATASET_CONFIG["drive_root"])
# New feature folder: YOLO-pose poses live alongside (not overwriting) AlphaPose poses.
DRIVE_POSE_TRAIN = DRIVE_ROOT / "pose_yolopose" / "train"
DRIVE_POSE_TEST = DRIVE_ROOT / "pose_yolopose" / "test"
DRIVE_LOG_DIR = DRIVE_ROOT / "logs"
GROUND_TRUTH_DIR = DATASET_CONFIG["ground_truth_dir"]
GT_REPO_SUBDIR = Path(DATASET_CONFIG["gt_repo_subdir"])
EXPECTED_TRAIN_CLIPS = int(DATASET_CONFIG["expected_train"])
EXPECTED_TEST_CLIPS = int(DATASET_CONFIG["expected_test"])

for directory in [LOCAL_POSE_WORK, DRIVE_POSE_TRAIN, DRIVE_POSE_TEST, DRIVE_LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Selected dataset:", DISPLAY_DATASET_NAME, f"({DATASET_KEY})")
print("STG-NF --dataset argument:", STG_NF_DATASET_ARG)
print("Repo target:", REPO_DIR)
print("Train source root:", TRAIN_SOURCE_ROOT)
print("Test source root:", TEST_SOURCE_ROOT)
print("Train source mode:", TRAIN_SOURCE_MODE)
print("Test source mode:", TEST_SOURCE_MODE)
print("Pose model:", POSE_MODEL_NAME)
print("DET_THR / TRACKER:", DET_THR, TRACKER)
print("Drive pose root:", DRIVE_ROOT)
if GROUND_TRUTH_DIR is not None:
    print("Ground-truth directory:", GROUND_TRUTH_DIR)


In [ ]:
ARCHIVE_PATH = Path(DATASET_CONFIG["archive_path"])
EXTRACT_ROOT = Path(DATASET_CONFIG["extract_root"])
EXTRACT_MARKER = Path(DATASET_CONFIG["extract_marker"])

if EXTRACT_MARKER.exists():
    print(f"Dataset already extracted: {EXTRACT_MARKER}")
else:
    assert ARCHIVE_PATH.exists(), f"Missing dataset archive: {ARCHIVE_PATH}"
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    if DATASET_CONFIG["archive_is_zip"]:
        print(f"Extracting {ARCHIVE_PATH} to {EXTRACT_ROOT} ...")
        !unzip -q "{ARCHIVE_PATH}" -d "{EXTRACT_ROOT}"
    else:
        print(f"Extracting {ARCHIVE_PATH} to {EXTRACT_ROOT} ...")
        !tar -xzvf "{ARCHIVE_PATH}" -C "{EXTRACT_ROOT}"

assert EXTRACT_MARKER.exists(), (
    f"Extraction finished but expected marker path is missing: {EXTRACT_MARKER}\n"
    f"Check the archive layout for {DISPLAY_DATASET_NAME}."
)
print("Dataset archive ready:", EXTRACT_MARKER)


## 2. Clone STG-NF Fork

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)
print("STG-NF ready at", REPO_DIR)


## 3. Install Ultralytics YOLO26-pose

`pip install ultralytics` is pure Python (no CUDA compilation); the `yolo26l-pose.pt`
weights auto-download on first use. `lap` is installed alongside so ByteTrack has its
linear-assignment dependency ready before the first `model.track()` call (no mid-run auto-install).


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install ultralytics lap

import ultralytics
print("ultralytics", ultralytics.__version__)


## 4. Load YOLO26-pose Model


In [ ]:
import torch
from ultralytics import YOLO

device = "cuda:0" if torch.cuda.is_available() else "cpu"

POSE_MODEL = YOLO(POSE_MODEL_NAME)
print("pose model ready:", POSE_MODEL_NAME)
print("device:", device)


## 4a. YOLO-pose Settings Check

In [ ]:
os.chdir(REPO_DIR)

print("YOLO-pose extraction settings:")
print("  pose model:", POSE_MODEL_NAME)
print("  DET_THR / TRACKER:", DET_THR, TRACKER)
print("  device:", device if "device" in globals() else "cuda:0")
print("\nYOLO-pose emits COCO-17 keypoints (same order AlphaPose uses);")
print("STG-NF's keypoints17_to_coco18 performs the 17->18 reorder downstream.")
print("\nYOLO-pose settings check complete.")


## 5. Locate Dataset Sources

In [ ]:
VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def clip_id_from_source(source, source_mode):
    source = Path(source)
    if DATASET_KEY == "avenue":
        # STG-NF expects scene_clip ids like 01_0005; Avenue videos are numbered 01..21.
        video_num = int(source.stem)
        return f"01_{video_num:04d}"
    return source.stem if source_mode == "video" else source.name

def list_video_sources(root):
    root = Path(root)
    assert root.exists(), f"Video root does not exist: {root}"
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS], key=lambda p: str(p))

def list_image_clip_dirs(root):
    root = Path(root)
    assert root.exists(), f"Image root does not exist: {root}"
    clip_dirs = []
    for dirpath, dirnames, filenames in os.walk(root):
        if any(Path(name).suffix.lower() in IMAGE_EXTS for name in filenames):
            clip_dirs.append(Path(dirpath))
    return sorted(set(clip_dirs), key=lambda p: str(p))

def list_sources(root, source_mode):
    if source_mode == "video":
        return list_video_sources(root)
    if source_mode == "images":
        return list_image_clip_dirs(root)
    raise ValueError(f"Unknown SOURCE_MODE: {source_mode}")

train_sources = list_sources(TRAIN_SOURCE_ROOT, source_mode=TRAIN_SOURCE_MODE)
test_sources = list_sources(TEST_SOURCE_ROOT, source_mode=TEST_SOURCE_MODE)
assert train_sources, f"No train sources found under {TRAIN_SOURCE_ROOT}"
assert test_sources, f"No test sources found under {TEST_SOURCE_ROOT}"

print("Train sources:", len(train_sources))
print("Test sources:", len(test_sources))
print("First train sources:", [str(p) for p in train_sources[:5]])
print("First test sources:", [str(p) for p in test_sources[:5]])
print("First train clip IDs:", [clip_id_from_source(p, source_mode=TRAIN_SOURCE_MODE) for p in train_sources[:5]])
print("First test clip IDs:", [clip_id_from_source(p, source_mode=TEST_SOURCE_MODE) for p in test_sources[:5]])


## 6. YOLO-pose Conversion and Resumable Extraction Helpers

In [ ]:
from tqdm import tqdm
import numpy as np
import torch
import cv2



def tracked_json_path(out_dir, clip_id):
    return Path(out_dir) / f"{clip_id}_yolopose_tracked_person.json"


def in_progress_path(out_dir, clip_id):
    return Path(out_dir) / f"{clip_id}.in_progress"


def manifest_path(out_dir):
    return Path(out_dir) / "pose_extraction_manifest.jsonl"


def is_json_readable(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with path.open("r") as handle:
            json.load(handle)
        return True
    except Exception:
        return False


def tracked_json_has_stg_format(path):
    path = Path(path)
    if not is_json_readable(path):
        return False
    with path.open("r") as handle:
        data = json.load(handle)
    if not isinstance(data, dict):
        return False
    if not data:
        return True
    first_person = next(iter(data.values()))
    if not isinstance(first_person, dict) or not first_person:
        return False
    first_record = next(iter(first_person.values()))
    return isinstance(first_record, dict) and "keypoints" in first_record and "scores" in first_record


def append_manifest(out_dir, record):
    path = manifest_path(out_dir)
    path.parent.mkdir(parents=True, exist_ok=True)
    record = dict(record)
    record["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S")
    with path.open("a") as handle:
        handle.write(json.dumps(record) + "\n")


# --- YOLO-pose -> STG-NF tracked-person JSON (COCO-17; no reorder, done by dataset.py) ---
def yolopose_to_tracked_person(keypoints, keypoint_scores, det_scores, track_ids, frame_indices, num_digits=4):
    tracked = {}
    for i in range(len(track_ids)):
        pid = str(int(track_ids[i]))
        fk = str(int(frame_indices[i])).zfill(num_digits)
        kp_flat = []
        for (x, y), c in zip(keypoints[i], keypoint_scores[i]):
            kp_flat.extend([float(x), float(y), float(c)])
        tracked.setdefault(pid, {})[fk] = {
            "keypoints": kp_flat,
            "scores": float(det_scores[i]),
        }
    return tracked


# --- frame iterator (video or image directory) ---
def iter_frames(source, source_mode):
    if source_mode == "images":
        files = sorted(
            [p for p in Path(source).iterdir()
             if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
        )
        for i, f in enumerate(files):
            img = cv2.imread(str(f))
            if img is not None:
                yield i, cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        cap = cv2.VideoCapture(str(source))
        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            yield i, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            i += 1
        cap.release()


# --- ByteTrack (Ultralytics built-in) replaces greedy IoU tracking ---
def _reset_pose_tracker():
    # Ultralytics keeps tracker state in POSE_MODEL.predictor.trackers. Reset it
    # before each clip so track IDs restart cleanly (IDs are per-clip in the JSON).
    try:
        for t in POSE_MODEL.predictor.trackers:
            t.reset()
    except Exception:
        pass


def run_yolopose_on_source(source, source_mode):
    # Returns the tracked-person dict for one clip.
    _reset_pose_tracker()
    all_kps, all_scores, all_det_scores, all_tids, all_frames = [], [], [], [], []
    for frame_idx, frame_rgb in iter_frames(source, source_mode):
        res = POSE_MODEL.track(frame_rgb, verbose=False, persist=True, tracker=TRACKER)[0]
        if res.boxes is None or len(res.boxes) == 0 or res.keypoints is None:
            continue
        confs = res.boxes.conf.cpu().numpy()
        keep = confs > DET_THR   # YOLO-pose is person-only, so confidence is the only filter
        if not keep.any():
            continue
        confs = confs[keep]                            # (M,) person-detection confidence
        kp = res.keypoints.xy.cpu().numpy()[keep]      # (M, 17, 2)
        kc = res.keypoints.conf.cpu().numpy()[keep]    # (M, 17)
        if res.boxes.id is not None:
            tids = res.boxes.id.cpu().numpy()[keep].astype(np.int64)
        else:
            tids = np.arange(len(confs), dtype=np.int64)  # fallback (rare): tracking unavailable
        for i in range(len(confs)):
            all_kps.append(kp[i])
            all_scores.append(kc[i])
            all_det_scores.append(float(confs[i]))
            all_tids.append(int(tids[i]))
            all_frames.append(frame_idx)
    if not all_kps:
        return {}
    keypoints = np.stack(all_kps)
    keypoint_scores = np.stack(all_scores)
    det_scores = np.asarray(all_det_scores, dtype=np.float64)
    track_ids = np.asarray(all_tids, dtype=np.int64)
    frame_indices = np.asarray(all_frames, dtype=np.int64)
    return yolopose_to_tracked_person(keypoints, keypoint_scores, det_scores, track_ids, frame_indices, num_digits=4)


def extract_one_source(source, drive_out_dir, split, source_mode, timeout=None, force=False):
    source = Path(source)
    drive_out_dir = Path(drive_out_dir)
    drive_out_dir.mkdir(parents=True, exist_ok=True)
    clip_id = clip_id_from_source(source, source_mode)
    drive_tracked = tracked_json_path(drive_out_dir, clip_id)
    marker = in_progress_path(drive_out_dir, clip_id)

    if not force and tracked_json_has_stg_format(drive_tracked):
        print(f"[skip] {clip_id}: tracked YOLO-pose JSON already exists in Drive")
        append_manifest(drive_out_dir, {"clip_id": clip_id, "source": str(source), "status": "skipped_complete"})
        return "skipped"

    marker.write_text(json.dumps({"source": str(source), "started": time.strftime("%Y-%m-%d %H:%M:%S")}))
    print(f"[run] {clip_id}")
    started = time.time()
    try:
        tracked = run_yolopose_on_source(source, source_mode)
        with drive_tracked.open("w") as handle:
            json.dump(tracked, handle)
        assert tracked_json_has_stg_format(drive_tracked), f"Bad tracked JSON after extraction: {drive_tracked}"
        elapsed = time.time() - started
        append_manifest(drive_out_dir, {
            "clip_id": clip_id,
            "source": str(source),
            "status": "processed",
            "elapsed_sec": round(elapsed, 2),
            "tracked": str(drive_tracked),
        })
        print(f"[done] {clip_id}: {elapsed / 60:.1f} min; tracked JSON copied to Drive")
        return "processed"
    except Exception as exc:
        append_manifest(drive_out_dir, {"clip_id": clip_id, "source": str(source), "status": "error", "error": repr(exc)})
        print(f"[error] {clip_id}: {exc}")
        raise
    finally:
        if marker.exists() and tracked_json_has_stg_format(drive_tracked):
            marker.unlink()


def extract_sources(sources, drive_out_dir, split, source_mode, limit=None, start=0, timeout=None, force=False, continue_on_error=True):
    selected = list(sources)[start:]
    if limit is not None:
        selected = selected[:limit]
    counts = {"processed": 0, "skipped": 0, "error": 0}
    for source in tqdm(selected, desc=f"extract -> {Path(drive_out_dir).name}"):
        try:
            result = extract_one_source(source, drive_out_dir, split=split, source_mode=source_mode, timeout=timeout, force=force)
            counts[result] = counts.get(result, 0) + 1
        except Exception as exc:
            counts["error"] += 1
            if not continue_on_error:
                raise
            print(f"[continue] {source}: {exc}")
    print("Extraction counts:", counts)
    return counts


print("YOLO-pose extraction helpers ready.")


## 7. Smoke Test One Train Clip and One Test Clip

In [ ]:
SMOKE_TIMEOUT = None  # Set seconds if you want a hard timeout.

smoke_train_source = train_sources[0]
smoke_test_source = test_sources[0]
print("Smoke train:", smoke_train_source, "->", clip_id_from_source(smoke_train_source, source_mode=TRAIN_SOURCE_MODE))
print("Smoke test:", smoke_test_source, "->", clip_id_from_source(smoke_test_source, source_mode=TEST_SOURCE_MODE))

extract_one_source(smoke_train_source, DRIVE_POSE_TRAIN, split="None", source_mode=TRAIN_SOURCE_MODE, timeout=SMOKE_TIMEOUT)
extract_one_source(smoke_test_source, DRIVE_POSE_TEST, split="None", source_mode=TEST_SOURCE_MODE, timeout=SMOKE_TIMEOUT)

for source, out_dir, mode in [(smoke_train_source, DRIVE_POSE_TRAIN, TRAIN_SOURCE_MODE), (smoke_test_source, DRIVE_POSE_TEST, TEST_SOURCE_MODE)]:
    clip_id = clip_id_from_source(source, source_mode=mode)
    assert tracked_json_has_stg_format(tracked_json_path(out_dir, clip_id)), tracked_json_path(out_dir, clip_id)
    print("validated", clip_id)


## 8. Full Resumable Extraction

Rerun this cell after a Colab restart. Completed clips are skipped based on Drive JSONs.

In [ ]:
BATCH_LIMIT = None  # Set an integer for smaller Colab sessions.
START_AT = 0
EXTRACTION_TIMEOUT = None
CONTINUE_ON_ERROR = True

extract_sources(train_sources, DRIVE_POSE_TRAIN, split="None", source_mode=TRAIN_SOURCE_MODE, limit=BATCH_LIMIT, start=START_AT, timeout=EXTRACTION_TIMEOUT, continue_on_error=CONTINUE_ON_ERROR)
extract_sources(test_sources, DRIVE_POSE_TEST, split="None", source_mode=TEST_SOURCE_MODE, limit=BATCH_LIMIT, start=START_AT, timeout=EXTRACTION_TIMEOUT, continue_on_error=CONTINUE_ON_ERROR)


## 9. Verify Drive Outputs

In [ ]:
def output_clip_ids(out_dir, suffix):
    out_dir = Path(out_dir)
    return {p.name[:-len(suffix)] for p in out_dir.glob(f"*{suffix}")}


def verify_outputs(sources, out_dir, source_mode):
    expected = {clip_id_from_source(s, source_mode=source_mode) for s in sources}
    tracked_ids = output_clip_ids(out_dir, "_yolopose_tracked_person.json")
    missing_tracked = sorted(expected - tracked_ids)
    extra_tracked = sorted(tracked_ids - expected)
    print(out_dir)
    print("  expected:", len(expected))
    print("  tracked:", len(tracked_ids))
    print("  missing tracked:", missing_tracked[:20], "count=", len(missing_tracked))
    print("  extra tracked:", extra_tracked[:20], "count=", len(extra_tracked))
    assert not missing_tracked, f"Missing tracked JSONs: {missing_tracked[:20]}"
    assert not extra_tracked, f"Unexpected tracked JSONs: {extra_tracked[:20]}"


verify_outputs(train_sources, DRIVE_POSE_TRAIN, source_mode=TRAIN_SOURCE_MODE)
verify_outputs(test_sources, DRIVE_POSE_TEST, source_mode=TEST_SOURCE_MODE)
print("All expected YOLO-pose tracked pose JSONs are present in Drive.")


## Prepare Ground Truth, Sanity Check, and Training (identical to STG-NF.ipynb)

In [ ]:
os.chdir(REPO_DIR)


def prepare_repo_ground_truth():
    gt_dst = REPO_DIR / GT_REPO_SUBDIR
    gt_dst.mkdir(parents=True, exist_ok=True)
    if DATASET_KEY == "shanghaitech":
        src_roots = [
            REPO_DIR / "data/ShanghaiTech/gt/test_frame_mask",
            Path("/content/shanghaitech/testing/test_frame_mask"),
        ]
        copied = 0
        for src_root in src_roots:
            if not src_root.exists():
                continue
            for src in src_root.glob("*.npy"):
                dst = gt_dst / src.name
                if not dst.exists():
                    shutil.copy2(src, dst)
                copied += 1
            if copied:
                break
        print(f"ShanghaiTech GT folder: {gt_dst} ({copied} files)")
        return gt_dst

    assert GROUND_TRUTH_DIR is not None, "GROUND_TRUTH_DIR is required for Avenue"
    assert Path(GROUND_TRUTH_DIR).exists(), f"Missing Avenue labels: {GROUND_TRUTH_DIR}"
    for video_num in range(1, 22):
        src = Path(GROUND_TRUTH_DIR) / f"{video_num}.npy"
        dst = gt_dst / f"01_{video_num:04d}.npy"
        if not src.exists():
            raise FileNotFoundError(f"Missing Avenue label file: {src}")
        shutil.copy2(src, dst)
    print("Prepared Avenue GT copies:", gt_dst)
    return gt_dst


prepare_repo_ground_truth()


In [ ]:
os.chdir(REPO_DIR)

# Reload repo modules after cloning the fork.
for name in list(sys.modules):
    if name == "dataset" or name == "args" or name.startswith("utils"):
        del sys.modules[name]

from args import init_parser, init_sub_args
from dataset import get_dataset_and_loader
from utils.data_utils import trans_list

argv = [
    "--dataset", STG_NF_DATASET_ARG,
    "--pose_path_train", str(DRIVE_POSE_TRAIN),
    "--pose_path_test", str(DRIVE_POSE_TEST),
    "--vid_path_train", str(TRAIN_SOURCE_ROOT),
    "--vid_path_test", str(TEST_SOURCE_ROOT),
    "--batch_size", "2",
    "--num_workers", "0",
    "--specific_clip", "0",
]
args = init_parser().parse_args(argv)
args, _ = init_sub_args(args)
dataset, loader = get_dataset_and_loader(args, trans_list=trans_list, only_test=False)
train_batch = next(iter(loader["train"]))
test_batch = next(iter(loader["test"]))
print("Train sample shape:", dataset["train"][0][0].shape)
print("Test sample shape:", dataset["test"][0][0].shape)
print("Train batch pose shape:", train_batch[0].shape)
print("Test batch pose shape:", test_batch[0].shape)

In [ ]:
# ── Attention configuration ────────────────────────────────────────────────
# Choose an attention mechanism to inject into the ST-GCN blocks.
#   'none'     : original STG-NF (no attention)
#   'dual'     : Dual-Attention (DAM) — skeleton + frame streams
#   'triplet'  : Triplet-Attention — skeleton + frame + channel streams
#   'skeleton' : skeleton-only attention
#   'frame'    : frame-only attention
#
# Regularization / capacity controls (see STG-NF README for details):
#   FREEZE_ATTENTION      : if True, attention params are initialized once
#                          (deterministic) and never updated. Reproduces
#                          the reference repo's accidental behaviour where
#                          attention acts as a fixed random projection.
#                          Recommended for small datasets.
#   ATTENTION_LR_MULT     : multiplier on the base LR for attention params
#                          (e.g. 0.1 = 10x slower). Only when trainable.
#   ATTENTION_WD_MULT     : multiplier on the base weight decay.
#   ATTENTION_PROJ_TYPE   : 'full' (reference, huge Linear) or 'bottleneck'
#                          (low-rank, far fewer params).
#   ATTENTION_BOTTLENECK  : bottleneck width (only for proj_type='bottleneck').
#
# These variables are consumed by the train_eval.py and stgnf_export_scores.py
# cells below. They must match between training and score export.
ATTENTION_TYPE = "none"   # one of: none, dual, triplet, skeleton, frame
N_HEADS = 1               # number of attention heads
N_MECATT = 1              # number of attention applications in series per st_gcn block
N_MECATT_INSIDE = 1       # number of inner attention iterations per application
FREEZE_ATTENTION = False  # freeze attention params (fixed random projection)
ATTENTION_LR_MULT = 0.1   # LR multiplier for attention (1.0 = same as base)
ATTENTION_WD_MULT = 5.0   # weight-decay multiplier for attention
ATTENTION_PROJ_TYPE = "bottleneck"        # 'full' or 'bottleneck'
ATTENTION_BOTTLENECK = 64            # bottleneck dim (proj_type='bottleneck' only)
print(f"Attention config: type={ATTENTION_TYPE} heads={N_HEADS} "
      f"n_mecatt={N_MECATT} n_mecatt_inside={N_MECATT_INSIDE} "
      f"freeze={FREEZE_ATTENTION} lr_mult={ATTENTION_LR_MULT} "
      f"wd_mult={ATTENTION_WD_MULT} proj={ATTENTION_PROJ_TYPE} "
      f"bottleneck={ATTENTION_BOTTLENECK}")


In [ ]:
BATCH_SIZE = 256
NUM_WORKERS = 2

os.chdir(REPO_DIR)
!python train_eval.py   --dataset {STG_NF_DATASET_ARG}   --checkpoint checkpoints/ShanghaiTech_85_9.tar   --pose_path_train "{DRIVE_POSE_TRAIN}"   --pose_path_test "{DRIVE_POSE_TEST}"   --vid_path_train "{TRAIN_SOURCE_ROOT}"   --vid_path_test "{TEST_SOURCE_ROOT}"   --batch_size {BATCH_SIZE}   --num_workers {NUM_WORKERS} --attention {ATTENTION_TYPE} --n_heads {N_HEADS} --n_mecatt {N_MECATT} --n_mecatt_inside {N_MECATT_INSIDE} {(' --freeze_attention' if FREEZE_ATTENTION else '')} --attention_lr_mult {ATTENTION_LR_MULT} --attention_wd_mult {ATTENTION_WD_MULT} --attention_proj_type {ATTENTION_PROJ_TYPE} --attention_bottleneck_dim {ATTENTION_BOTTLENECK}


In [ ]:
os.chdir(REPO_DIR)
!python train_eval.py   --dataset {STG_NF_DATASET_ARG}   --pose_path_train "{DRIVE_POSE_TRAIN}"   --pose_path_test "{DRIVE_POSE_TEST}"   --vid_path_train "{TRAIN_SOURCE_ROOT}"   --vid_path_test "{TEST_SOURCE_ROOT}"   --exp_dir "{DRIVE_LOG_DIR}"   --epochs 8   --batch_size 256   --num_workers 2 --attention {ATTENTION_TYPE} --n_heads {N_HEADS} --n_mecatt {N_MECATT} --n_mecatt_inside {N_MECATT_INSIDE} {(' --freeze_attention' if FREEZE_ATTENTION else '')} --attention_lr_mult {ATTENTION_LR_MULT} --attention_wd_mult {ATTENTION_WD_MULT} --attention_proj_type {ATTENTION_PROJ_TYPE} --attention_bottleneck_dim {ATTENTION_BOTTLENECK}


## Export STG-NF Frame-Level Scores for the Ensemble

The following cells run `stgnf_export_scores.py` from the cloned STG-NF fork
to dump raw frame-level anomaly scores for every test video in the selected
dataset. The output is `stgnf_scores.pkl`, a dictionary keyed by `video_id`
whose values hold parallel arrays of `(frame_index, anomaly_score)`. This
format matches the MULDE export produced in the MULDE training notebook so the
PRISM script can load both with the same code path.



In [ ]:
os.chdir(REPO_DIR)
BATCH_SIZE = 256
NUM_WORKERS = 2

export_script_path = REPO_DIR / "stgnf_export_scores.py"
assert export_script_path.exists(), f"Missing export script in cloned repo: {export_script_path}"
print(f"Using repo export script: {export_script_path}")


In [ ]:
STGNF_OUTPUT_PKL = Path(DRIVE_LOG_DIR) / "stgnf_scores.pkl"
STGNF_OUTPUT_PKL.parent.mkdir(parents=True, exist_ok=True)
# Ensure this path is correct and accessible
STG_NF_CHECKPOINT_PATH = None  # Set to your trained checkpoint .pth.tar path after training

os.chdir(REPO_DIR)
!python stgnf_export_scores.py   --dataset {STG_NF_DATASET_ARG}   --checkpoint "{STG_NF_CHECKPOINT_PATH}"   --pose_path_train "{DRIVE_POSE_TRAIN}"   --pose_path_test "{DRIVE_POSE_TEST}"   --vid_path_train "{TRAIN_SOURCE_ROOT}"   --vid_path_test "{TEST_SOURCE_ROOT}"   --output_pkl {STGNF_OUTPUT_PKL}   --batch_size {BATCH_SIZE}   --num_workers {NUM_WORKERS} --attention {ATTENTION_TYPE} --n_heads {N_HEADS} --n_mecatt {N_MECATT} --n_mecatt_inside {N_MECATT_INSIDE} {(' --freeze_attention' if FREEZE_ATTENTION else '')} --attention_lr_mult {ATTENTION_LR_MULT} --attention_wd_mult {ATTENTION_WD_MULT} --attention_proj_type {ATTENTION_PROJ_TYPE} --attention_bottleneck_dim {ATTENTION_BOTTLENECK}

print(f"STG-NF scores saved to: {STGNF_OUTPUT_PKL}")
if STGNF_OUTPUT_PKL.exists():
    print(f"Success! Size: {STGNF_OUTPUT_PKL.stat().st_size} bytes")
else:
    print("Failed to generate scores file.")
